In [1]:
# ODIN — Board Report Generator (Monthly / Quarterly)
# CEO & Board-Ready. No Noise. No Excuses.

"""
WHAT THIS MODULE DOES
---------------------
- Generates a BOARD-READY PDF (monthly / quarterly)
- Uses ODIN agents as the single source of truth
- Produces narrative + metrics + risks
- Zero dashboards. Zero slides. One document.

This is what boards actually read.
"""

# ================================================================
# IMPORTS
# ================================================================
import os
import xmlrpc.client
import pandas as pd
from typing import Dict, Any
from dotenv import load_dotenv

from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors

from langchain_openai import ChatOpenAI

# ================================================================
# CONFIG
# ================================================================
load_dotenv()

# Initialize with your credentials
ODOO_URL = os.getenv('ODOO_URL', 'https://vendyltd.odoo.com/')
ODOO_DB = os.getenv('ODOO_DB', 'vendyltd')
ODOO_USER = os.getenv('ODOO_USER', 'muktadir@vendy.ltd')
ODOO_PASSWORD = os.getenv('ODOO_PASSWORD', '205958c9bb841b4bef7e6da4a3afb5b5029cd6ae')

LLM_MODEL = "gpt-4o-mini"
llm = ChatOpenAI(model=LLM_MODEL, temperature=0)

# ================================================================
# ODOO CLIENT (READ ONLY)
# ================================================================
class OdooClient:
    def __init__(self):
        common = xmlrpc.client.ServerProxy(f"{ODOO_URL}/xmlrpc/2/common")
        self.uid = common.authenticate(ODOO_DB, ODOO_USER, ODOO_PASSWORD, {})
        self.models = xmlrpc.client.ServerProxy(f"{ODOO_URL}/xmlrpc/2/object")

    def read(self, model, domain=None, fields=None):
        return self.models.execute_kw(
            ODOO_DB,
            self.uid,
            ODOO_PASSWORD,
            model,
            'search_read',
            [domain or []],
            {'fields': fields or []}
        )

odoo = OdooClient()

# ================================================================
# SAFETY
# ================================================================

def safe_df(records, cols):
    if not records:
        return pd.DataFrame(columns=cols)
    df = pd.DataFrame(records)
    for c in cols:
        if c not in df.columns:
            df[c] = 0
    return df

# ================================================================
# ODIN CORE METRICS (BOARD LEVEL ONLY)
# ================================================================

def board_metrics(date_from=None, date_to=None) -> Dict[str, Any]:
    domain = []
    if date_from:
        domain.append(('date', '>=', date_from))
    if date_to:
        domain.append(('date', '<=', date_to))

    df = safe_df(
        odoo.read(
            'account.move',
            domain=domain,
            fields=['amount_total', 'move_type', 'state', 'payment_state']
        ),
        ['amount_total', 'move_type', 'state', 'payment_state']
    )

    df['amount_total'] = df['amount_total'].astype(float)

    revenue = df[(df['move_type'] == 'out_invoice') & (df['state'] == 'posted')]['amount_total'].sum()
    expenses = df[(df['move_type'] == 'in_invoice') & (df['state'] == 'posted')]['amount_total'].sum()

    cash_in = df[(df['move_type'] == 'out_invoice') & (df['payment_state'] == 'paid')]['amount_total'].sum()
    cash_out = df[(df['move_type'] == 'in_invoice') & (df['payment_state'] == 'paid')]['amount_total'].sum()

    ar = df[(df['move_type'] == 'out_invoice') & (df['payment_state'] != 'paid')]['amount_total'].sum()
    ap = df[(df['move_type'] == 'in_invoice') & (df['payment_state'] != 'paid')]['amount_total'].sum()

    return {
        "Revenue": round(revenue, 2),
        "Expenses": round(expenses, 2),
        "Profit": round(revenue - expenses, 2),
        "Cash Flow": round(cash_in - cash_out, 2),
        "Accounts Receivable": round(ar, 2),
        "Accounts Payable": round(ap, 2)
    }

# ================================================================
# BOARD NARRATIVE (AI, BUT CONTROLLED)
# ================================================================

def board_narrative(metrics: Dict[str, Any]) -> str:
    prompt = f"""
You are preparing a board report.
No hype. No fluff. No excuses.

METRICS:
{metrics}

Write:
- Executive summary (5–6 lines)
- Financial health assessment
- Top 3 risks
- Top 3 actions next quarter
"""

    return llm.invoke(prompt).content

# ================================================================
# PDF GENERATOR (BOARD FORMAT)
# ================================================================

def generate_board_pdf(metrics: Dict[str, Any], narrative: str, period: str):
    file_name = f"ODIN_Board_Report_{period}.pdf"

    doc = SimpleDocTemplate(file_name, pagesize=A4)
    styles = getSampleStyleSheet()
    story = []

    story.append(Paragraph("<b>ODIN — Board Report</b>", styles['Title']))
    story.append(Paragraph(f"Period: {period}", styles['Normal']))
    story.append(Spacer(1, 12))

    table_data = [[k, v] for k, v in metrics.items()]
    table = Table(table_data, colWidths=[250, 200])
    table.setStyle(TableStyle([
        ('BACKGROUND', (0,0), (-1,0), colors.grey),
        ('GRID', (0,0), (-1,-1), 1, colors.black)
    ]))

    story.append(Paragraph("<b>Financial Snapshot</b>", styles['Heading2']))
    story.append(table)
    story.append(Spacer(1, 20))

    story.append(Paragraph("<b>Executive Narrative</b>", styles['Heading2']))
    for line in narrative.split("\n"):
        story.append(Paragraph(line, styles['Normal']))
        story.append(Spacer(1, 6))

    doc.build(story)
    return file_name

# ================================================================
# ENTRY
# ================================================================
if __name__ == "__main__":
    print("ODIN — Board Report Generator")

    period = input("Enter period (e.g. 2025-Q1 or 2025-03): ").strip()
    date_from = input("Date from (YYYY-MM-DD): ").strip()
    date_to = input("Date to (YYYY-MM-DD): ").strip()

    metrics = board_metrics(date_from or None, date_to or None)
    narrative = board_narrative(metrics)

    pdf = generate_board_pdf(metrics, narrative, period)
    print(f"\nBoard report generated: {pdf}")

ODIN — Board Report Generator
Enter period (e.g. 2025-Q1 or 2025-03): 2025-Q1
Date from (YYYY-MM-DD): 2025-08-01
Date to (YYYY-MM-DD): 2025-12-31

Board report generated: ODIN_Board_Report_2025-Q1.pdf
